# Model 1: Conventional Conv-LSTM Baseline

TensorFlow/Keras conventional convolutional LSTM baseline for five-class Bonn EEG seizure-related classification.

This GitHub-ready notebook has cleared execution outputs for lightweight version control. Run the cells sequentially after mounting or configuring the Bonn EEG dataset path used in the project.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import random
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_style("whitegrid")

print("✅ Imports loaded successfully")
print("TensorFlow version:", tf.__version__)

In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# --- CELL 2: LOAD BONN EEG DATA (5-CLASS, FULL RECORDINGS) ---

base_path = "/kaggle/input/datasets/quands/eeg-dataset/Dataset/"
class_names = ['F', 'N', 'O', 'S', 'Z']

signals = []
labels = []
file_names = []

for folder in class_names:
    folder_path = os.path.join(base_path, folder)
    txt_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(".txt")])

    print(f"Reading {folder}: {len(txt_files)} files")

    for file in txt_files:
        file_path = os.path.join(folder_path, file)

        with open(file_path, "r") as f:
            values = f.read().split()
            signal = np.array([float(x) for x in values], dtype=np.float32)

        signals.append(signal)
        labels.append(folder)
        file_names.append(file)

X = np.array(signals, dtype=np.float32)
y = np.array(labels)
file_names = np.array(file_names)

print("\n✅ Data loading complete")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))
print("\nClass distribution:")
print(pd.Series(y).value_counts().sort_index())
print("\nSignal length check:", np.unique([len(sig) for sig in X]))

In [ ]:
# --- CELL 4: LABEL ENCODING + TRAIN/VAL/TEST SPLIT + CHUNKING ---

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label mapping:")
for original, encoded in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{original} -> {encoded}")

# Split full recordings first: train / val / test
X_train_full, X_temp_full, y_train_full, y_temp_full, files_train, files_temp = train_test_split(
    X, y_encoded, file_names,
    test_size=0.3,
    random_state=SEED,
    stratify=y_encoded
)

X_val_full, X_test_full, y_val_full, y_test_full, files_val, files_test = train_test_split(
    X_temp_full, y_temp_full, files_temp,
    test_size=2/3,   # final = 70% train, 10% val, 20% test
    random_state=SEED,
    stratify=y_temp_full
)

print("\nOriginal split complete (before chunking)")
print("X_train_full shape:", X_train_full.shape)
print("X_val_full shape:", X_val_full.shape)
print("X_test_full shape:", X_test_full.shape)

# -------- Chunking function --------
def chunk_signals(signals, labels, file_ids, chunk_size=256, stride=128):
    X_chunks = []
    y_chunks = []
    file_chunks = []

    for signal, label, fid in zip(signals, labels, file_ids):
        signal_length = len(signal)

        for start in range(0, signal_length - chunk_size + 1, stride):
            end = start + chunk_size
            chunk = signal[start:end]
            X_chunks.append(chunk)
            y_chunks.append(label)
            file_chunks.append(fid)

    return (
        np.array(X_chunks, dtype=np.float32),
        np.array(y_chunks),
        np.array(file_chunks)
    )

# -------- Apply chunking AFTER split --------
chunk_size = 256
stride = 128

X_train, y_train, train_chunk_files = chunk_signals(
    X_train_full, y_train_full, files_train, chunk_size=chunk_size, stride=stride
)
X_val, y_val, val_chunk_files = chunk_signals(
    X_val_full, y_val_full, files_val, chunk_size=chunk_size, stride=stride
)
X_test, y_test, test_chunk_files = chunk_signals(
    X_test_full, y_test_full, files_test, chunk_size=chunk_size, stride=stride
)

# Optional but recommended: normalize each chunk
def zscore_per_chunk(X, eps=1e-8):
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + eps)

X_train = zscore_per_chunk(X_train)
X_val   = zscore_per_chunk(X_val)
X_test  = zscore_per_chunk(X_test)

# Reshape for model input: (samples, timesteps, features)
X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print("\nChunked dataset shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)
print(f"\nChunk size: {chunk_size}, Stride: {stride}")

In [ ]:
# --- CELL 5: BUILD THE HYBRID CONV-LSTM MODEL ---

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Dropout

model = Sequential([
    Input(shape=(chunk_size, 1)),

    Conv1D(32, kernel_size=7, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(64, kernel_size=5, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    LSTM(128, return_sequences=True),
    Dropout(0.3),

    LSTM(64, return_sequences=False),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(5, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

In [ ]:
# --- CELL 6: TRAIN THE MODEL ---

from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import time


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights:", class_weights)

print("🚀 Starting chunked 5-class Conv-LSTM training...")

train_start = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[reduce_lr],
    class_weight=class_weights,
    verbose=1
)

train_end = time.time()
training_time_seconds = train_end - train_start

print(f"\nTotal Training Time: {training_time_seconds:.2f} seconds")

In [ ]:
# --- CELL 7: MULTICLASS EVALUATION + TRAINING PLOTS + EFFICIENCY METRICS ---

# 0) Training history plots
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(train_acc) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, train_loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 1) Total training time
training_time_seconds = sum(history.history.get('time', [])) if 'time' in history.history else None

# Fallback if per-epoch timing is not stored
if training_time_seconds is None or training_time_seconds == 0:
    print("Note: Per-epoch timing was not stored in history. Use external timing around model.fit() for exact training time.")

# 2) Inference latency
start_time = time.time()
y_pred_probs = model.predict(X_test, verbose=0)
end_time = time.time()

latency_per_sample = (end_time - start_time) / len(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 3) Classification metrics
test_acc = accuracy_score(y_test, y_pred)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='macro', zero_division=0
)

print("=" * 60)
print("5-CLASS STANDARD LSTM EVALUATION")
print("=" * 60)
print(f"Accuracy:                  {test_acc:.4f}")
print(f"Macro Precision:           {precision_macro:.4f}")
print(f"Macro Recall:              {recall_macro:.4f}")
print(f"Macro F1-score:            {f1_macro:.4f}")

# 4) Parameter count
total_params = model.count_params()
print(f"Total Parameters:          {total_params:,}")

# 5) Weight sparsity
zero_params = 0
for layer in model.layers:
    for weights in layer.get_weights():
        zero_params += np.count_nonzero(weights == 0)

weight_sparsity = (zero_params / total_params) * 100 if total_params > 0 else 0
print(f"Weight Sparsity:           {weight_sparsity:.4f}%")

# 6) Activation sparsity proxy
activation_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.layers[-2].output
)
dense_activations = activation_model.predict(X_test, verbose=0)

eps = 1e-3
activation_sparsity = np.mean(np.abs(dense_activations) < eps) * 100
print(f"Activation Sparsity Proxy: {activation_sparsity:.4f}%")

# 7) Inference latency
print(f"Inference Latency:         {latency_per_sample * 1000:.4f} ms/sample")

# 8) Total training time
if training_time_seconds is not None and training_time_seconds > 0:
    print(f"Total Training Time:       {training_time_seconds:.2f} s")
    print(f"Total Training Time:       {training_time_seconds / 60:.2f} min")

print("=" * 60)

# 9) Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

# 10) Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("5-Class Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# --- CELL 8: SIGNAL-LEVEL EVALUATION (SAME FORMAT AS CHUNK-LEVEL) ---

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1) Predict on chunked test set
start_time = time.time()
y_pred_probs = model.predict(X_test, verbose=0)
end_time = time.time()

# 2) Sanity checks
print("len(test_chunk_files):", len(test_chunk_files))
print("len(y_test):", len(y_test))
print("len(y_pred_probs):", len(y_pred_probs))

# 3) Build dataframe linking each chunk prediction to its original signal
df_pred = pd.DataFrame({
    "file": test_chunk_files,
    "true": y_test
})

for i in range(y_pred_probs.shape[1]):
    df_pred[f"p_{i}"] = y_pred_probs[:, i]

# 4) Check that each original file has only one true label
label_check = df_pred.groupby("file")["true"].nunique()
print("Files with >1 true label:", (label_check > 1).sum())

# 5) Aggregate chunk probabilities into signal-level probabilities
prob_cols = [f"p_{i}" for i in range(y_pred_probs.shape[1])]
signal_probs_df = df_pred.groupby("file")[prob_cols].mean()
signal_true_df = df_pred.groupby("file")["true"].first()

signal_true = signal_true_df.values
signal_pred_probs = signal_probs_df.values
signal_pred = np.argmax(signal_pred_probs, axis=1)

# 6) Metrics
signal_acc = accuracy_score(signal_true, signal_pred)
signal_macro_precision = precision_score(signal_true, signal_pred, average='macro', zero_division=0)
signal_macro_recall = recall_score(signal_true, signal_pred, average='macro', zero_division=0)
signal_macro_f1 = f1_score(signal_true, signal_pred, average='macro', zero_division=0)

num_signals = len(signal_true)
signal_latency_ms = ((end_time - start_time) / num_signals) * 1000

print("\n" + "="*70)
print("SIGNAL-LEVEL EVALUATION (AGGREGATED FROM CHUNKS)")
print("="*70)
print(f"Accuracy:                 {signal_acc:.4f}")
print(f"Macro Precision:          {signal_macro_precision:.4f}")
print(f"Macro Recall:             {signal_macro_recall:.4f}")
print(f"Macro F1-score:           {signal_macro_f1:.4f}")
print(f"Inference Latency:        {signal_latency_ms:.4f} ms/signal")
print("="*70)

# 7) Classification report
print("\nSignal-Level Classification Report:\n")
print(classification_report(
    signal_true,
    signal_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

# 8) Confusion matrix
cm_signal = confusion_matrix(signal_true, signal_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_signal,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("Signal-Level 5-Class Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# --- CELL 9: FLOPs + CO2 EMISSIONS ESTIMATION ---

import time
import numpy as np
import tensorflow as tf

# -----------------------------
# 1) FLOPs estimation
# -----------------------------
def get_model_flops(model, input_shape):
    """
    Estimate FLOPs for a single forward pass.
    input_shape should exclude batch dimension, e.g. (256, 1)
    """
    @tf.function
    def model_fn(x):
        return model(x, training=False)

    concrete_func = model_fn.get_concrete_function(
        tf.TensorSpec([1, *input_shape], tf.float32)
    )

    frozen_func, graph_def = convert_variables_to_constants_v2_as_graph(concrete_func)

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd="op",
            options=opts
        )

    return flops.total_float_ops if flops is not None else 0


from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph

input_shape = (chunk_size, 1)
total_flops = get_model_flops(model, input_shape)

print("=" * 70)
print("MODEL COMPUTE EFFICIENCY")
print("=" * 70)
print(f"Input Shape:              {input_shape}")
print(f"Total Parameters:         {model.count_params():,}")
print(f"Estimated FLOPs:          {total_flops:,} per sample")
print(f"Estimated MFLOPs:         {total_flops / 1e6:.3f} MFLOPs/sample")
print("=" * 70)



In [ ]:
# --- CELL 10: CO2 ESTIMATION ---

training_time_seconds = 246.15   # replace if you retrain again

avg_gpu_power_watts = 70        # Tesla T4 typical draw estimate
grid_carbon_intensity = 0.475   # kg CO2 per kWh

energy_kwh = (avg_gpu_power_watts * training_time_seconds) / 1000 / 3600
co2_kg = energy_kwh * grid_carbon_intensity
co2_g = co2_kg * 1000

print("=" * 70)
print("TRAINING ENERGY / CO2 ESTIMATION")
print("=" * 70)
print(f"Training Time:            {training_time_seconds:.2f} s")
print(f"Assumed GPU Power Draw:   {avg_gpu_power_watts:.1f} W")
print(f"Energy Used:              {energy_kwh:.8f} kWh")
print(f"Carbon Intensity:         {grid_carbon_intensity:.3f} kg CO2/kWh")
print(f"Estimated CO2 Emissions:  {co2_kg:.8f} kg CO2")
print(f"Estimated CO2 Emissions:  {co2_g:.4f} g CO2")
print("=" * 70)

# Manuscript Extensions for Model 1

These appended cells implement the reviewer-facing tasks for the conventional Conv-LSTM baseline:

1. Baseline diagnostic figure on a Bonn Set E/S seizure signal. This model has no temporal attention, so the lower trace is chunk-level confidence/evidence rather than an attention map.
2. AWGN wearable-noise stress testing at 20 dB, 10 dB, and 0 dB.
3. Correct/incorrect prediction export and pairwise McNemar testing hooks.
4. Hardware deployment constraints: memory, theoretical energy, and CPU latency.

Run these cells after the original training/evaluation cells so `model`, `X_test_full`, `y_test_full`, `files_test`, and `label_encoder` are already defined.


In [ ]:
# =========================================================
# CELL 11: MANUSCRIPT TASK SETUP - MODEL 1
# MANUSCRIPT_TASKS_MODEL1_APPEND_V1
# =========================================================

import os
import time
import math
import itertools
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

try:
    from scipy.stats import binomtest
except Exception:
    binomtest = None

if os.path.exists('/kaggle/working'):
    MANUSCRIPT_DIR_MODEL1 = Path('/kaggle/working/manuscript_model1')
else:
    MANUSCRIPT_DIR_MODEL1 = Path.cwd() / 'manuscript_model1'

MANUSCRIPT_DIR_MODEL1.mkdir(parents=True, exist_ok=True)

BONN_SAMPLING_RATE_HZ = 4097 / 23.6
SNR_LEVELS_DB = [20, 10, 0]
MODEL1_NAME = 'Model 1 Conventional Conv-LSTM'

model1_keras_path = MANUSCRIPT_DIR_MODEL1 / 'model1_current_for_manuscript.keras'
model.save(model1_keras_path)

print('=' * 80)
print('MODEL 1 MANUSCRIPT TASK SETUP')
print('=' * 80)
print(f'Output directory: {MANUSCRIPT_DIR_MODEL1}')
print(f'Keras checkpoint saved: {model1_keras_path}')
print(f'Bonn sampling rate: {BONN_SAMPLING_RATE_HZ:.4f} Hz')
print(f'SNR stress-test levels: {SNR_LEVELS_DB}')
print('Note: Model 1 has no temporal attention; Task 1 uses chunk confidence as a baseline diagnostic.')


def manuscript1_savefig(filename, dpi=600):
    path = MANUSCRIPT_DIR_MODEL1 / filename
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Saved: {path}')
    return path


def model1_label_name(label_idx):
    return label_encoder.classes_[int(label_idx)]


def model1_normalize_to_unit(x):
    x = np.asarray(x, dtype=np.float64)
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x, dtype=np.float64)
    return (x - lo) / (hi - lo)


def model1_chunk_one_signal(signal, chunk_size=chunk_size, stride=stride):
    chunks = []
    starts = []
    for start in range(0, len(signal) - chunk_size + 1, stride):
        chunks.append(signal[start:start + chunk_size])
        starts.append(start)
    return np.asarray(chunks, dtype=np.float32), np.asarray(starts, dtype=np.int64)


def model1_signal_to_tensor(signal):
    chunks, starts = model1_chunk_one_signal(signal, chunk_size=chunk_size, stride=stride)
    chunks_norm = zscore_per_chunk(chunks)
    return chunks_norm[..., np.newaxis].astype(np.float32), chunks_norm, starts


def select_model1_bonn_set_e_signal(preferred_splits=('test', 'val', 'train'), seizure_index=0):
    s_idx = int(np.where(label_encoder.classes_ == 'S')[0][0])
    for split_name in preferred_splits:
        X_name = f'X_{split_name}_full'
        y_name = f'y_{split_name}_full'
        files_name = f'files_{split_name}'
        if X_name not in globals() or y_name not in globals():
            continue
        X_split = globals()[X_name]
        y_split = globals()[y_name]
        f_split = globals().get(files_name, np.array([''] * len(y_split)))
        matches = np.where(np.asarray(y_split) == s_idx)[0]
        if len(matches) > 0:
            pos = int(matches[seizure_index % len(matches)])
            return X_split[pos], int(y_split[pos]), str(f_split[pos]), split_name, pos

    raise ValueError('No Bonn Set E/S seizure signal found in available arrays.')


def model1_overlap_add(values_by_chunk, starts, signal_len, value_width):
    timeline = np.zeros(signal_len, dtype=np.float64)
    counts = np.zeros(signal_len, dtype=np.float64)
    for chunk_idx, chunk_start in enumerate(starts):
        seg_end = min(int(chunk_start) + value_width, signal_len)
        usable = seg_end - int(chunk_start)
        if usable <= 0:
            continue
        values = np.asarray(values_by_chunk[chunk_idx], dtype=np.float64)
        if values.ndim == 0:
            timeline[int(chunk_start):seg_end] += float(values)
        else:
            timeline[int(chunk_start):seg_end] += values[:usable]
        counts[int(chunk_start):seg_end] += 1.0
    valid = counts > 0
    timeline[valid] /= counts[valid]
    return timeline


In [ ]:
# =========================================================
# TASK 1: BASELINE DIAGNOSTIC FIGURE - MODEL 1 HAS NO ATTENTION
# =========================================================


def create_model1_baseline_diagnostic_figure(
    signal=None,
    label=None,
    seizure_index=0,
    preferred_splits=('test', 'val', 'train'),
    filename_prefix='task1_model1_baseline_diagnostic',
):
    """
    Creates a vertically stacked baseline figure:
    1. Raw continuous EEG waveform.
    2. Normalized absolute-amplitude event proxy.
    3. Chunk-level model confidence for the predicted class.

    This baseline has no temporal attention mechanism, so the third panel is not an attention map.
    """
    if signal is None or label is None:
        signal, label, file_name, source_split, source_pos = select_model1_bonn_set_e_signal(
            preferred_splits=preferred_splits,
            seizure_index=seizure_index,
        )
    else:
        file_name, source_split, source_pos = '', 'manual', seizure_index

    signal = np.asarray(signal, dtype=np.float32)
    chunks_tensor, chunks_norm, starts = model1_signal_to_tensor(signal)
    probs = model.predict(chunks_tensor, verbose=0)
    avg_probs = probs.mean(axis=0)
    pred_idx = int(np.argmax(avg_probs))

    confidence_by_chunk = probs[:, pred_idx]
    confidence_trace = model1_overlap_add(confidence_by_chunk, starts, len(signal), chunk_size)

    # Conventional baseline proxy: high-amplitude activity in the normalized chunks.
    amp_proxy_chunks = np.abs(chunks_norm)
    amp_proxy_trace = model1_overlap_add(amp_proxy_chunks, starts, len(signal), chunk_size)
    amp_proxy_unit = model1_normalize_to_unit(amp_proxy_trace)
    confidence_unit = model1_normalize_to_unit(confidence_trace)

    time_axis = np.arange(len(signal)) / BONN_SAMPLING_RATE_HZ
    true_name = model1_label_name(label)
    pred_name = model1_label_name(pred_idx)

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(13, 8.2),
        sharex=True,
        gridspec_kw={'height_ratios': [2.0, 1.15, 1.15]},
    )

    axes[0].plot(time_axis, signal, color='#111827', linewidth=0.8)
    axes[0].set_ylabel('Raw EEG')
    axes[0].set_title(
        f'Model 1 Baseline Evidence on Bonn Set E/S Seizure '
        f'(source={source_split}[{source_pos}], file={file_name}, true={true_name}, pred={pred_name})'
    )
    axes[0].grid(True, alpha=0.18)

    axes[1].plot(time_axis, amp_proxy_unit, color='#2563eb', linewidth=1.0)
    axes[1].fill_between(time_axis, 0, amp_proxy_unit, color='#bfdbfe', alpha=0.55)
    axes[1].set_ylabel('Amplitude proxy\nnormalized')
    axes[1].set_ylim(0, 1.02)
    axes[1].grid(True, alpha=0.18)

    axes[2].plot(time_axis, confidence_unit, color='#be123c', linewidth=1.35)
    axes[2].fill_between(time_axis, 0, confidence_unit, color='#fecdd3', alpha=0.55)
    axes[2].set_ylabel('Predicted-class\nconfidence')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylim(0, 1.02)
    axes[2].grid(True, alpha=0.18)

    fig.tight_layout()
    png_path = manuscript1_savefig(f'{filename_prefix}.png')
    pdf_path = MANUSCRIPT_DIR_MODEL1 / f'{filename_prefix}.pdf'
    fig.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    trace_df = pd.DataFrame({
        'time_sec': time_axis,
        'raw_eeg': signal,
        'amplitude_proxy': amp_proxy_trace,
        'amplitude_proxy_normalized': amp_proxy_unit,
        'predicted_class_confidence': confidence_trace,
        'predicted_class_confidence_normalized': confidence_unit,
    })
    trace_path = MANUSCRIPT_DIR_MODEL1 / f'{filename_prefix}_traces.csv'
    trace_df.to_csv(trace_path, index=False)
    print(f'Saved traces: {trace_path}')

    return {
        'figure_png': png_path,
        'figure_pdf': pdf_path,
        'trace_csv': trace_path,
        'true_label': true_name,
        'pred_label': pred_name,
        'source_split': source_split,
        'source_index': source_pos,
        'file_name': file_name,
        'probabilities': avg_probs,
    }


task1_model1_baseline_result = create_model1_baseline_diagnostic_figure(seizure_index=0)
task1_model1_baseline_result


In [ ]:
# =========================================================
# TASK 2: WEARABLE NOISE-ROBUSTNESS STRESS TEST - MODEL 1 AWGN
# =========================================================


def inject_awgn_model1(signals, snr_db, seed=SEED):
    rng = np.random.default_rng(seed)
    signals = np.asarray(signals, dtype=np.float32)
    noisy = np.empty_like(signals, dtype=np.float32)

    for i, signal in enumerate(signals):
        signal_power = float(np.mean(signal ** 2))
        noise_power = signal_power / (10.0 ** (float(snr_db) / 10.0))
        noise = rng.normal(0.0, math.sqrt(noise_power), size=signal.shape).astype(np.float32)
        noisy[i] = signal + noise

    return noisy


def model1_evaluate_signal_level(X_signals, y_signals, files=None, aggregation='avg_probs'):
    rows = []
    targets = []
    preds = []
    probs_all = []
    signal_times = []
    chunks_per_signal = []

    if files is None:
        files = np.array([''] * len(y_signals))

    # Warm-up.
    warm_x, _, _ = model1_signal_to_tensor(X_signals[0])
    _ = model.predict(warm_x, verbose=0)

    start_total = time.time()
    for i, (signal, label, file_name) in enumerate(zip(X_signals, y_signals, files)):
        chunks_tensor, _, _ = model1_signal_to_tensor(signal)
        start_signal = time.time()
        probs = model.predict(chunks_tensor, verbose=0)
        signal_times.append(time.time() - start_signal)

        chunk_preds = probs.argmax(axis=1)
        avg_probs = probs.mean(axis=0)

        if aggregation == 'majority':
            pred = int(np.bincount(chunk_preds, minlength=len(label_encoder.classes_)).argmax())
        elif aggregation == 'avg_probs':
            pred = int(np.argmax(avg_probs))
        else:
            raise ValueError('aggregation must be one of: majority, avg_probs')

        targets.append(int(label))
        preds.append(pred)
        probs_all.append(avg_probs)
        chunks_per_signal.append(int(chunks_tensor.shape[0]))
        rows.append({
            'Signal_Index': i,
            'File': str(file_name),
            'True_Label_Index': int(label),
            'True_Label': model1_label_name(label),
            'Pred_Label_Index': int(pred),
            'Pred_Label': model1_label_name(pred),
            'Correct': int(int(label) == pred),
        })

    total_time = time.time() - start_total
    targets = np.asarray(targets, dtype=np.int64)
    preds = np.asarray(preds, dtype=np.int64)
    probs_all = np.vstack(probs_all)

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets,
        preds,
        average='macro',
        zero_division=0,
    )

    metrics = {
        'targets': targets,
        'preds': preds,
        'avg_probs': probs_all,
        'correct': (targets == preds).astype(np.int64),
        'acc': float(accuracy_score(targets, preds)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'avg_latency_per_signal_sec': float(np.mean(signal_times)),
        'avg_chunks_per_signal': float(np.mean(chunks_per_signal)),
        'avg_latency_per_chunk_sec': float(np.mean(signal_times) / np.mean(chunks_per_signal)),
        'total_time_sec': float(total_time),
        'total_chunks': int(np.sum(chunks_per_signal)),
    }
    return pd.DataFrame(rows), metrics


def run_model1_awgn_stress_test(snrs_db=SNR_LEVELS_DB, aggregation='avg_probs'):
    rows = []

    for snr_db in snrs_db:
        X_test_noisy = inject_awgn_model1(X_test_full, snr_db=snr_db, seed=SEED + int(100 * float(snr_db)))
        _, metrics = model1_evaluate_signal_level(
            X_test_noisy,
            y_test_full,
            files=files_test if 'files_test' in globals() else None,
            aggregation=aggregation,
        )

        rows.append({
            'Model': MODEL1_NAME,
            'SNR_dB': float(snr_db),
            'Signal_Accuracy': float(metrics['acc']),
            'Signal_Macro_F1': float(metrics['f1']),
            'Macro_Precision': float(metrics['precision']),
            'Macro_Recall': float(metrics['recall']),
            'Input_Spike_Rate': np.nan,
            'Hidden_Spike_Rate': np.nan,
        })

        print(f'SNR {snr_db:>5} dB | Acc={metrics["acc"]:.4f} | F1={metrics["f1"]:.4f}')

    df = pd.DataFrame(rows).sort_values('SNR_dB', ascending=False)
    csv_path = MANUSCRIPT_DIR_MODEL1 / 'task2_model1_awgn_robustness.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

    plot_df = df.sort_values('SNR_dB')
    plt.figure(figsize=(7.2, 4.6))
    plt.plot(plot_df['SNR_dB'], plot_df['Signal_Accuracy'], marker='o', linewidth=2.0, label=MODEL1_NAME)
    plt.xlabel('SNR (dB)')
    plt.ylabel('Classification Accuracy')
    plt.title('Model 1 Noise Robustness Under AWGN')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.25)
    plt.legend(frameon=False)
    plt.tight_layout()
    manuscript1_savefig('task2_model1_accuracy_vs_snr.png')
    pdf_path = MANUSCRIPT_DIR_MODEL1 / 'task2_model1_accuracy_vs_snr.pdf'
    plt.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    return df


task2_model1_noise_df = run_model1_awgn_stress_test()
task2_model1_noise_df


In [ ]:
# =========================================================
# TASK 3: STATISTICAL SIGNIFICANCE - MODEL 1 PREDICTION EXPORT + MCNEMAR HOOKS
# =========================================================


def export_model1_signal_predictions(aggregation='avg_probs'):
    files = files_test if 'files_test' in globals() else None
    df, metrics = model1_evaluate_signal_level(X_test_full, y_test_full, files=files, aggregation=aggregation)
    df.insert(0, 'Model', MODEL1_NAME)

    for class_idx, class_name in enumerate(label_encoder.classes_):
        df[f'Prob_{class_name}'] = metrics['avg_probs'][:, class_idx]

    csv_path = MANUSCRIPT_DIR_MODEL1 / 'task3_model1_signal_predictions_correctness.csv'
    df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    print(f'Model 1 clean signal accuracy: {metrics["acc"]:.4f}')
    print(f'Model 1 clean signal macro-F1: {metrics["f1"]:.4f}')

    checkpoint_path = MANUSCRIPT_DIR_MODEL1 / f'model1_acc_{metrics["acc"]:.4f}.keras'
    model.save(checkpoint_path)
    print(f'Saved locked Model 1 Keras checkpoint: {checkpoint_path}')

    metadata = {
        'model': MODEL1_NAME,
        'test_acc': metrics['acc'],
        'test_f1': metrics['f1'],
        'chunk_size': int(chunk_size),
        'stride': int(stride),
        'normalization': 'per_chunk_zscore',
        'class_names': list(label_encoder.classes_),
        'checkpoint': str(checkpoint_path),
    }
    metadata_path = MANUSCRIPT_DIR_MODEL1 / f'model1_acc_{metrics["acc"]:.4f}_metadata.json'
    import json
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(f'Saved metadata: {metadata_path}')

    return df, metrics


def mcnemar_exact_or_corrected_model1(correct_a, correct_b):
    a = np.asarray(correct_a).astype(bool)
    b = np.asarray(correct_b).astype(bool)
    a_correct_b_wrong = int(np.sum(a & ~b))
    a_wrong_b_correct = int(np.sum(~a & b))
    discordant = a_correct_b_wrong + a_wrong_b_correct

    if discordant == 0:
        return {
            'A_Correct_B_Wrong': a_correct_b_wrong,
            'A_Wrong_B_Correct': a_wrong_b_correct,
            'Discordant_Pairs': discordant,
            'P_Value': 1.0,
            'Method': 'no_discordant_pairs',
        }

    if binomtest is not None:
        p_value = float(
            binomtest(
                min(a_correct_b_wrong, a_wrong_b_correct),
                n=discordant,
                p=0.5,
                alternative='two-sided',
            ).pvalue
        )
        method = 'exact_binomial_mcnemar'
    else:
        statistic = (abs(a_correct_b_wrong - a_wrong_b_correct) - 1.0) ** 2 / discordant
        p_value = float(math.erfc(math.sqrt(statistic / 2.0)))
        method = 'chi_square_continuity_mcnemar'

    return {
        'A_Correct_B_Wrong': a_correct_b_wrong,
        'A_Wrong_B_Correct': a_wrong_b_correct,
        'Discordant_Pairs': discordant,
        'P_Value': p_value,
        'Method': method,
    }


def run_pairwise_mcnemar_from_prediction_csvs_model1(named_csv_paths, merge_on='File'):
    """
    Compare any number of model prediction CSVs.
    Prefer merge_on='File' when all CSVs include filenames; otherwise use 'Signal_Index'.
    """
    tables = {}
    for name, csv_path in named_csv_paths.items():
        df = pd.read_csv(csv_path)
        if merge_on in df.columns:
            df = df.sort_values(merge_on).reset_index(drop=True)
        else:
            df = df.sort_values('Signal_Index').reset_index(drop=True)
        tables[name] = df

    rows = []
    for name_a, name_b in itertools.combinations(tables.keys(), 2):
        a = tables[name_a]
        b = tables[name_b]
        key = merge_on if merge_on in a.columns and merge_on in b.columns else 'Signal_Index'
        merged = a[[key, 'True_Label_Index', 'Correct']].merge(
            b[[key, 'True_Label_Index', 'Correct']],
            on=[key, 'True_Label_Index'],
            suffixes=('_A', '_B'),
        )
        result = mcnemar_exact_or_corrected_model1(merged['Correct_A'], merged['Correct_B'])
        result.update({
            'Model_A': name_a,
            'Model_B': name_b,
            'Merge_Key': key,
            'Aligned_Test_Signals': len(merged),
            'Significant_p_lt_0_05': bool(result['P_Value'] < 0.05),
        })
        rows.append(result)

    out_df = pd.DataFrame(rows)
    csv_path = MANUSCRIPT_DIR_MODEL1 / 'task3_pairwise_mcnemar_results.csv'
    out_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')
    return out_df


task3_model1_predictions_df, task3_model1_clean_metrics = export_model1_signal_predictions()
task3_model1_predictions_df.head()


In [ ]:
# =========================================================
# TASK 4: COMPREHENSIVE HARDWARE PROFILING - MODEL 1
# =========================================================


def model1_parameter_memory_table(model):
    total_params = int(model.count_params())
    return pd.DataFrame([{
        'Model': MODEL1_NAME,
        'Total_Parameters': total_params,
        'Trainable_Parameters': total_params,
        'Float32_Memory_KB': total_params * 4 / 1024,
        'Float32_Memory_MB': total_params * 4 / (1024 ** 2),
        'Int8_Quantized_Memory_KB': total_params / 1024,
        'One_Bit_Theoretical_Memory_KB': np.nan,
    }])


def estimate_model1_mac_counts():
    """Approximate dense MAC/AC counts per 256-sample chunk for the Conv-LSTM baseline."""
    B = 1
    T = int(chunk_size)
    macs = 0
    acs = 0

    # Conv1D 1->32, k=7, same padding, output length 256.
    macs += B * T * 32 * 1 * 7
    acs += B * T * 32
    # MaxPool to 128.
    acs += B * 128 * 32

    # Conv1D 32->64, k=5, output length 128.
    macs += B * 128 * 64 * 32 * 5
    acs += B * 128 * 64
    # MaxPool to 64.
    acs += B * 64 * 64

    # LSTM 64 input features -> 128 hidden, return sequences over 64 steps.
    # Per step: xW + hU for 4 gates.
    macs += B * 64 * (64 * 4 * 128 + 128 * 4 * 128)
    acs += B * 64 * 128 * 40

    # LSTM 128 input features -> 64 hidden over 64 steps.
    macs += B * 64 * (128 * 4 * 64 + 64 * 4 * 64)
    acs += B * 64 * 64 * 40

    # Dense 64->128 and 128->5.
    macs += B * 64 * 128
    acs += B * 128
    macs += B * 128 * 5

    return {
        'Dense_MACs_Per_Chunk': float(macs),
        'Dense_ACs_Per_Chunk': float(acs),
        'Spike_Aware_MACs_Per_Chunk': np.nan,
        'Spike_Aware_ACs_Per_Chunk': np.nan,
        'Input_Spike_Rate_Used': np.nan,
        'Hidden_Spike_Rate_Used': np.nan,
    }


def measure_model1_cpu_latency_per_chunk_ms(repeats=100, warmup=20):
    original_visible = None
    x = np.zeros((1, chunk_size, 1), dtype=np.float32)
    for _ in range(warmup):
        _ = model.predict(x, verbose=0)
    start = time.perf_counter()
    for _ in range(repeats):
        _ = model.predict(x, verbose=0)
    elapsed = time.perf_counter() - start
    return (elapsed / repeats) * 1000.0


def run_model1_hardware_profile(mac_energy_pj=4.6, ac_energy_pj=0.9, repeats=100, warmup=20):
    mem_df = model1_parameter_memory_table(model)

    if 'task3_model1_clean_metrics' not in globals():
        _, clean_metrics = export_model1_signal_predictions()
    else:
        clean_metrics = task3_model1_clean_metrics

    ops = estimate_model1_mac_counts()
    cpu_latency_ms = measure_model1_cpu_latency_per_chunk_ms(repeats=repeats, warmup=warmup)
    avg_chunks_per_signal = float(clean_metrics.get('avg_chunks_per_signal', np.nan))

    dense_energy_j = (ops['Dense_MACs_Per_Chunk'] * mac_energy_pj + ops['Dense_ACs_Per_Chunk'] * ac_energy_pj) * 1e-12

    hw_row = mem_df.iloc[0].to_dict()
    hw_row.update(ops)
    hw_row.update({
        'MAC_Energy_pJ': float(mac_energy_pj),
        'AC_Energy_pJ': float(ac_energy_pj),
        'Dense_Energy_J_Per_Chunk': float(dense_energy_j),
        'Spike_Aware_Energy_J_Per_Chunk': np.nan,
        'CPU_Latency_ms_Per_Chunk': float(cpu_latency_ms),
        'Avg_Chunks_Per_Signal': avg_chunks_per_signal,
        'CPU_Latency_ms_Per_Signal_Est': float(cpu_latency_ms * avg_chunks_per_signal),
        'Dense_Energy_J_Per_Signal_Est': float(dense_energy_j * avg_chunks_per_signal),
    })

    hw_df = pd.DataFrame([hw_row])
    csv_path = MANUSCRIPT_DIR_MODEL1 / 'task4_model1_hardware_deployment_constraints.csv'
    hw_df.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

    display_cols = [
        'Model',
        'Total_Parameters',
        'Float32_Memory_KB',
        'Int8_Quantized_Memory_KB',
        'Dense_Energy_J_Per_Chunk',
        'CPU_Latency_ms_Per_Chunk',
        'CPU_Latency_ms_Per_Signal_Est',
    ]

    fig, ax = plt.subplots(figsize=(13, 2.8))
    ax.axis('off')
    ax.set_title('Hardware Deployment Constraints - Model 1', fontsize=13, pad=10)

    table_df = hw_df[display_cols].copy()
    for col in table_df.columns:
        if col != 'Model':
            table_df[col] = table_df[col].map(lambda x: 'N/A' if pd.isna(x) else f'{x:.6g}')

    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7.3)
    table.scale(1, 1.45)
    plt.tight_layout()
    manuscript1_savefig('task4_model1_hardware_deployment_constraints.png')
    pdf_path = MANUSCRIPT_DIR_MODEL1 / 'task4_model1_hardware_deployment_constraints.pdf'
    plt.savefig(pdf_path, bbox_inches='tight')
    print(f'Saved: {pdf_path}')
    plt.show()

    return hw_df


task4_model1_hardware_df = run_model1_hardware_profile(repeats=100, warmup=20)
task4_model1_hardware_df
